In [10]:
# Q1
import re
text = "강의는 2026-05-06, 숙제는 2026-05-18에 마감"
print(re.match(r"\d+", text))
print(re.search(r"\d{4}-\d{2}-\d{2}", text).group())
print(re.findall(r"\d{4}-\d{2}-\d{2}", text))
print(re.findall(r"(\d{4})-(\d{2})-(\d{2})", text))
print(re.findall(r"(?:\d{4})-(?:\d{2})-(?:\d{2})", text))

None
2026-05-06
['2026-05-06', '2026-05-18']
[('2026', '05', '06'), ('2026', '05', '18')]
['2026-05-06', '2026-05-18']


(1) 
(a) None
(b) 2026-05-06
(c) ['2026-05-06', '2026-05-18']
(d) [('2026', '05', '06'), ('2026', '05', '18')]
(e) ['2026-05-06', '2026-05-18']

(2)
re.findall은 패턴에 괄호로 묶인 캡처 그룹이 있으면 전체 매칭 문자열을 버리고 그룹 안에 일치한 부분만 꺼내서 돌려주기 때문입니다. 그래서 (d)는 그룹이 3개라 튜플 리스트로 나오고, (e)의 (?:)는 비캡처 그룹이라 기억을 하지 않으므로 그룹이 없는 (c)와 똑같이 전체 문자열 리스트로 나오는 것입니다.

In [ ]:
# Q2
import re
html = "<b>안녕</b> <i>세상</i>!"
nums = "수강생 30명, 조교 3명"
print(re.sub(r"<.+>", "[T]", html))
print(re.sub(r"<.+?>", "[T]", html))
print(re.sub(r"<[^>]+>", "[T]", html))
print(re.sub(r"(\d+)", r"<\1>", nums))
print(re.sub(r"(\d+)", "<\1>", nums))

[T]!
[T]안녕[T] [T]세상[T]!
[T]안녕[T] [T]세상[T]!
수강생 <30>명, 조교 <3>명
수강생 <>명, 조교 <>명


(1) 
(a) [T]!
(b) [T]안녕[T] [T]세상[T]!
(c) [T]안녕[T] [T]세상[T]!
(d) 수강생 <30>명, 조교 <3>명
(e) 수강생 <r>명, 조교 <r>명

(2)
(i) (a)에 사용된 .+는 탐욕적 수량자이기 때문에 첫 번째 '<'부터 마지막'>'까지 통째로 치환하지만 , (b)의 .+?는 게으른 수량자라 각 태그의 닫는 괄호까지만 최소한으로 쪼개어 개별적으로 치환하기 때문입니다.
(ii) 원시 문자열 기호 r을 빼놓으면 파이썬이 \1을 정규표현식 엔진의 그룹 역참조가 아니라 8진수 이스케이프 문자(\x01)로 먼저 해석해서 잘못된 문자열을 전달하기 때문입니다.

In [19]:
import re
from collections import Counter

URL_PATTERN = re.compile(r"https?://\S+")
HTML_PATTERN = re.compile(r"<[^>]+>")
EMAIL_PATTERN = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_PATTERN = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#\w+")
JAMO_PATTERN = re.compile(r"[\u3131-\u3163]+")
WHITESPACE_PATTERN = re.compile(r"\s+")

def clean_post(post: str) -> str:
    post = URL_PATTERN.sub(" ", post)
    post = HTML_PATTERN.sub("", post)
    post = EMAIL_PATTERN.sub("[이메일]", post)
    post = PHONE_PATTERN.sub("[전화]", post)
    post = MENTION_PATTERN.sub(" ", post)
    post = HASHTAG_PATTERN.sub(" ", post)
    post = JAMO_PATTERN.sub("", post)
    post = WHITESPACE_PATTERN.sub(" ", post)
    return post.strip()

def extract_hashtags(post: str) -> list[str]:
    return re.findall(r"#(\w+)", post)

def analyze_posts(posts: list[str]) -> dict:
    posts_n: int = len(posts)
    total_length: int = 0
    all_hashtags: list[str] = []
    total_masked: int = 0
    
    for post in posts:
        all_hashtags.extend(extract_hashtags(post))
        
        p = URL_PATTERN.sub(" ", post)
        p = HTML_PATTERN.sub("", p)
        
        p, email_cnt = EMAIL_PATTERN.subn("[이메일]", p)
        p, phone_cnt = PHONE_PATTERN.subn("[전화]", p)
        total_masked += (email_cnt + phone_cnt)
        
        p = MENTION_PATTERN.sub(" ", p)
        p = HASHTAG_PATTERN.sub(" ", p)
        p = JAMO_PATTERN.sub("", p)
        p = WHITESPACE_PATTERN.sub(" ", p).strip()
        
        total_length += len(p)
        
    avg_length_after_clean: float = round(total_length / posts_n, 2) if posts_n > 0 else 0.0
    
    counter = Counter(all_hashtags)
    hashtag_counts: dict[str, int] = dict(counter.most_common())
    
    return {
        "posts_n": posts_n,
        "avg_length_after_clean": avg_length_after_clean,
        "hashtag_counts": hashtag_counts,
        "masked_count": total_masked
    }

posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ",
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠㅠㅠㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강.",
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로！"
]

for post in posts:
    print(clean_post(post))

print(analyze_posts(posts))

오늘 수업 진짜 재밌었음!! 감사
자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15강.
문의는 [이메일] ([전화])로！
{'posts_n': 5, 'avg_length_after_clean': 14.4, 'hashtag_counts': {'파이썬': 1, 'DCCP2026': 1, '팀플': 1}, 'masked_count': 2}


(1)
오늘 수업 진짜 재밌었음!! 감사
자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15강.
문의는 [이메일] ([전화])로！

(2)
{'posts_n': 5, 'avg_length_after_clean': 14.4, 'hashtag_counts': {'파이썬': 1, 'DCCP2026': 1, '팀플': 1}, 'masked_count': 2}

(설명)
만약 3단계(마스킹)보다 4단계(멘션 제거)를 먼저 처리해 버리면, mam3b@snu.ac.kr 이메일 주소 한가운데에 있는 @snu 부분이 멘션 패턴인 @\w+에 잡혀서 공백으로 먼저 지워지게 됩니다. 이로 인해 이메일 주소 모양이 완전히 깨져버려서 정작 3단계로 돌아왔을 때 이메일을 마스킹하지 못합니다.